# Natural disaster risk

Frequency & distribution score across six hazards on the 0.5° atlas grid, built from the Harvard Dataverse **"Global Multihazard Frequency and Distribution"** dataset (Dilley et al., DOI [10.7910/DVN/JFJNNU](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/JFJNNU)). Each raster pixel carries a categorical `VALUE` code; the sibling DBF maps that code to 0–10 scores for `flood`, `cyclone`, `drought`, `landslide`, `earthquake`, and `volcano`.

We join raster to DBF, coarsen each hazard onto the atlas grid, then weight and normalize via `common.compute_natural_disaster_risk` (defaults from `NATURAL_DISASTER_DEFAULT_WEIGHTS`). The final layer is sign-inverted so higher = safer, and ocean cells are masked using `is_land` from `grid.nc`. Wildfire is not included in this source and is silently omitted from the combination. The mortality-weighted sibling (`gdmhzmrt.zip`) is intentionally not used yet.

In [ ]:
import zipfile

import numpy as np
import pandas as pd
import xarray as xr
from dbfread import DBF

from common import (
    PROCESSED_DIR,
    RAW_DIR,
    compute_natural_disaster_risk,
    load_grid,
    plot_map,
    save_variable,
)

VARIABLE = 'natural_disaster_risk'
variable_raw = RAW_DIR / VARIABLE
variable_raw.mkdir(parents=True, exist_ok=True)

## 1. Fetch raw data

Manual download — grab the "Global Multihazard Frequency and Distribution" archive from https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/JFJNNU and place `gdmhz.zip` under `data/world-livable-atlas/raw/natural_disaster_risk/`. The cell below extracts it in place if needed.

The archive ships an ESRI ASCII raster (`gdmhz.asc`, ~175 MB, WGS84) whose pixel values are categorical `VALUE` codes, plus a dBase attribute table (`gdmhz.dbf`) that maps each code to per-hazard frequency scores (0–10).

In [ ]:
zip_path = variable_raw / 'gdmhz.zip'
extracted_dir = variable_raw / 'gdmhz'
asc_path = extracted_dir / 'gdmhz.asc'
dbf_path = extracted_dir / 'gdmhz.dbf'

if not zip_path.exists():
    raise FileNotFoundError(
        f'{zip_path} is missing — download from '
        'https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/JFJNNU'
    )

if not (asc_path.exists() and dbf_path.exists()):
    print(f'extracting {zip_path.name}...')
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(variable_raw)

print(f'asc: {asc_path.stat().st_size / 1024**2:.1f} MB')
print(f'dbf: {dbf_path.stat().st_size / 1024:.1f} KB')

## 2. Join raster to hazards and aggregate onto the grid

Each raster pixel is a `VALUE` code; the DBF has one row per code with 0–10 frequency scores per hazard. For each hazard we build a lookup array indexed by `VALUE`, apply it vectorised to every pixel to get a per-hazard raster, then coarsen (~12×12 native cells per 0.5° block) and interpolate onto the shared atlas grid.

CHRR's raster is 95% NODATA (`-9999`) — the source classifies a pixel only when at least one hazard is present, so NODATA means "no significant hazard" rather than "unmeasured". We therefore substitute `0` for NODATA before aggregating, otherwise ~84% of land cells would end up NaN. Ocean cells still coarsen to 0 but are stripped by the `is_land` mask in §4.

In [ ]:
# ASC header — matches the values in the file header (verified against gdmhz.asc)
NCOLS, NROWS = 8633, 3430
XLL, YLL = -179.9984, -58.0161
CELLSIZE = 0.0417

HAZARD_COLS = {
    'D3FLD': 'flood',
    'D3CYC': 'cyclone',
    'D3DRG': 'drought',
    'D3LND': 'landslide',
    'D3PGA': 'earthquake',
    'D3VOL': 'volcano',
}

# VALUE -> per-hazard score (0-10) lookup table
dbf_df = (
    pd.DataFrame(iter(DBF(str(dbf_path))))
    .rename(columns=HAZARD_COLS)
    .set_index('VALUE')[list(HAZARD_COLS.values())]
    .astype('float32')
)

# Load the raster of VALUE codes (~30 M pixels, one row per line)
print(f'loading raster ({NROWS}x{NCOLS} int values) — takes ~1 min...')
raster = np.loadtxt(asc_path, skiprows=6, dtype=np.int32)
assert raster.shape == (NROWS, NCOLS), raster.shape

# Row 0 in the ASC is the northernmost row; flip so lat is ascending for xarray
raster = raster[::-1, :]
lat = YLL + (np.arange(NROWS) + 0.5) * CELLSIZE
lon = XLL + (np.arange(NCOLS) + 0.5) * CELLSIZE

grid = load_grid()
lat_factor = round(float(abs(grid.lat[1] - grid.lat[0])) / CELLSIZE)
lon_factor = round(float(abs(grid.lon[1] - grid.lon[0])) / CELLSIZE)

max_value = int(dbf_df.index.max())
valid = (raster >= 0) & (raster <= max_value)

# NODATA (-9999) means "no significant hazard" in this source, so substitute 0
# instead of NaN before coarsening. Unknown VALUE codes (present in raster but
# not in DBF) also fall back to 0.
hazards = {}
for name in dbf_df.columns:
    lookup = np.zeros(max_value + 1, dtype='float32')
    lookup[dbf_df.index.to_numpy()] = dbf_df[name].to_numpy()
    scores = np.where(valid, lookup[np.clip(raster, 0, max_value)], 0.0)
    da = xr.DataArray(scores, coords={'lat': lat, 'lon': lon}, dims=('lat', 'lon'))
    hazards[name] = (
        da.coarsen(lat=lat_factor, lon=lon_factor, boundary='trim').mean()
        .interp(lat=grid.lat, lon=grid.lon, method='nearest')
        .astype('float32')
        .rename(name)
    )

hazards_ds = xr.Dataset(hazards)
hazards_ds

## 3. Cache intermediate

Save the per-hazard stack so `compute_natural_disaster_risk` can be called later (e.g. from a Dash app or scoring notebook) with custom weights, without re-fetching or re-cleaning. Mirrors the `_apparent_temp_monthly.nc` cache written by `14_temperature_pleasantness.ipynb`.

In [ ]:
cache_path = PROCESSED_DIR / '_hazards_by_type.nc'
if cache_path.exists():
    cache_path.unlink()
hazards_ds.to_netcdf(cache_path)
print(f'wrote {cache_path}')

## 4. Combine, sign-invert, and mask

`compute_natural_disaster_risk` normalizes each hazard to `[0, 1]` and takes a weighted average using the defaults from `common.NATURAL_DISASTER_DEFAULT_WEIGHTS` (earthquake / cyclone / flood at 1.0, wildfire / landslide at 0.5, drought at 0.7, volcano at 0.3). We invert the sign so higher = safer, matching the atlas convention, and mask ocean cells.

In [ ]:
risk = compute_natural_disaster_risk(cache_path)
values = (-risk).astype('float32').rename(VARIABLE).where(grid.is_land == 1)
values

## 5. Plot

In [ ]:
plot_map(values, cmap='RdYlGn', robust=True)

## 6. Save

In [ ]:
out = save_variable(values, VARIABLE)
print(f'wrote {out}')